# SECCIÓN 1 - Introducción
En este notebook se construyen las variables supervisadas necesarias para el modelado del M5 Forecasting.<br>
El objetivo es generar un conjunto de features que capture:<br>

dependencias temporales (lags)

tendencias y estacionalidad (ventanas móviles)

información de calendario (día, mes, año, día de la semana)

efectos de eventos y SNAP

relaciones entre ventas y tiempo

Para evitar problemas de memoria durante el desarrollo, se utiliza una muestra representativa del dataset limpio generado en el Notebook 02.<br>
El dataset completo se procesará posteriormente en el pipeline final.<br>
El resultado de este notebook será un conjunto de features listo para modelado, almacenado en:<br>

**data/features/exploratory/m5_features_sample.parquet**

# SECCIÓN 2 - Carga del dataset sample

In [10]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../../data/processed/exploratory/m5_clean_sample.parquet")

print(df.shape)
df.head()

(95650, 22)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_100_CA_1_validation,HOBBIES_1_100,HOBBIES_1,HOBBIES,CA_1,CA,d_1,1,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,10.00
1,HOBBIES_1_287_CA_1_validation,HOBBIES_1_287,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,5.88
2,HOUSEHOLD_1_202_CA_1_validation,HOUSEHOLD_1_202,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,1.98
3,HOUSEHOLD_1_537_CA_1_validation,HOUSEHOLD_1_537,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,15.98
4,FOODS_2_382_CA_1_validation,FOODS_2_382,FOODS_2,FOODS,CA_1,CA,d_1,2,2011-01-29,11101,...,1,2011,None,None,None,None,0,0,0,3.25


# SECCIÓN 3 - Preparación inicial
## SECCIÓN 3.1 - Conversión de tipos

In [3]:
df["date"] = pd.to_datetime(df["date"])

## SECCIÓN 3.2 - Ordenación por serie temporal

In [4]:
df = df.sort_values(["id", "date"], kind="mergesort").reset_index(drop=True)

## SECCIÓN 3.3 - Validación del orden temporal

In [5]:
assert (df.groupby("id")["date"].apply(lambda x: x.is_monotonic_increasing).all())

## SECCIÓN 3.4 - Revisión de valores nulos

In [6]:
df.isna().mean().sort_values(ascending=False).head(10)

event_type_2    0.997909
event_name_2    0.997909
event_type_1    0.919498
event_name_1    0.919498
id              0.000000
item_id         0.000000
state_id        0.000000
store_id        0.000000
cat_id          0.000000
dept_id         0.000000
dtype: float64

# SECCIÓN 4 - Generación de variables lag
Los lags permiten capturar dependencias temporales entre ventas pasadas y futuras.

In [8]:
LAG_DAYS = [1, 7, 28]

for lag in LAG_DAYS:
    df[f"lag_{lag}"] = df.groupby("id")["sales"].shift(lag)

## SECCIÓN 4.1 - Comprobación

In [9]:
df[["sales", "lag_1", "lag_7", "lag_28"]].head(20)

,sales,lag_1,lag_7,lag_28
0,0,NaN,NaN,NaN
1,1,0.0,NaN,NaN
2,0,1.0,NaN,NaN
3,0,0.0,NaN,NaN
4,4,0.0,NaN,NaN
5,0,4.0,NaN,NaN
6,1,0.0,NaN,NaN
7,2,1.0,0.0,NaN
8,0,2.0,1.0,NaN
9,0,0.0,0.0,NaN


Los primeros valores son NaN porque no hay datos previos.

A partir de la fila 7 empiezan a aparecer valores en lag_7.

lag_28 aparece más tarde, como es lógico.

# SECCIÓN 5 - Ventanas móviles (rolling windows)
Se aplican ventanas móviles calculadas únicamente sobre observaciones pasadas.<br>
Para evitar data leakage, las ventanas se calculan sobre valores desplazados 28 días hacia atrás, garantizando que ninguna información futura sea utilizada.

In [10]:
WINDOWS = [7, 28]

for w in WINDOWS:
    df[f"rolling_mean_{w}"] = (
        df.groupby("id")["sales"]
          .shift(28)
          .rolling(w)
          .mean()
    )
    df[f"rolling_std_{w}"] = (
        df.groupby("id")["sales"]
          .shift(28)
          .rolling(w)
          .std()
    )

## SECCIÓN 5.1 - Comprobación

In [11]:
df[["sales", "rolling_mean_7", "rolling_std_7"]].head(20)

,sales,rolling_mean_7,rolling_std_7
0,0,NaN,NaN
1,1,NaN,NaN
2,0,NaN,NaN
3,0,NaN,NaN
4,4,NaN,NaN
5,0,NaN,NaN
6,1,NaN,NaN
7,2,NaN,NaN
8,0,NaN,NaN
9,0,NaN,NaN


# SECCIÓN 6 - Features de calendario

In [12]:
df["dow"] = df["date"].dt.weekday
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["day"] = df["date"].dt.day

# SECCIÓN 7 - Eventos y SNAP

In [13]:
df["is_event_1"] = df["event_name_1"].notna().astype(int)
df["is_event_2"] = df["event_name_2"].notna().astype(int)

# SECCIÓN 7.1 - Comprobación

In [14]:
df[["is_event_1", "is_event_2"]].sum()

is_event_1    7700
is_event_2     200
dtype: int64

# SECCIÓN 8 - Codificación categórica

In [15]:
from sklearn.preprocessing import LabelEncoder

for col in ["item_id", "dept_id", "cat_id", "store_id", "state_id"]:
    df[col] = LabelEncoder().fit_transform(df[col])

La codificación categórica se aplica en esta fase únicamente para validar la compatibilidad con modelos supervisados en pandas.<br>
En el pipeline final implementado en Spark, las variables categóricas se gestionan mediante soporte nativo del modelo.

# SECCIÓN 9 - Validaciones finales
## SECCIÓN 9.1 Tamaño del dataset

In [16]:
df.shape

(95650, 34)

Se han añadido 12 columnas nuevas (lags, rolling, calendario, eventos).

## SECCIÓN 9.2 - Revisión de lags

In [17]:
df[["sales", "lag_1", "lag_7", "lag_28"]].head(40)

,sales,lag_1,lag_7,lag_28
0,0,NaN,NaN,NaN
1,1,0.0,NaN,NaN
2,0,1.0,NaN,NaN
3,0,0.0,NaN,NaN
4,4,0.0,NaN,NaN
5,0,4.0,NaN,NaN
6,1,0.0,NaN,NaN
7,2,1.0,0.0,NaN
8,0,2.0,1.0,NaN
9,0,0.0,0.0,NaN


## SECCIÓN 9.3 - Revisión de rolling

In [18]:
df[["sales", "rolling_mean_7", "rolling_mean_28"]].head(40)

,sales,rolling_mean_7,rolling_mean_28
0,0,NaN,NaN
1,1,NaN,NaN
2,0,NaN,NaN
3,0,NaN,NaN
4,4,NaN,NaN
5,0,NaN,NaN
6,1,NaN,NaN
7,2,NaN,NaN
8,0,NaN,NaN
9,0,NaN,NaN


# SECCIÓN 10 - Guardado del dataset de features

In [19]:
os.makedirs("data/features/exploratory", exist_ok=True)
df.to_parquet("data/features/exploratory/m5_features_sample.parquet", index=False)

print("Features sample guardadas correctamente.")

Features sample guardadas correctamente.


# SECCIÓN 10 — Limitaciones detectadas durante el Feature Engineering
Durante las pruebas realizadas sobre el dataset completo, las operaciones de generación de variables temporales (lags y rolling windows) mostraron un elevado consumo de memoria y tiempos de ejecución significativamente altos.<br>
En particular, las operaciones groupby combinadas con ventanas móviles sobre más de 58 millones de registros excedieron la capacidad de procesamiento disponible en el entorno local.<br>
Por este motivo, se decidió utilizar un subconjunto representativo del dataset para validar la lógica de generación de features.<br>
Las transformaciones desarrolladas en este notebook constituyen la base conceptual que posteriormente será implementada en un pipeline distribuido utilizando Apache Spark.

# SECCIÓN 11 — Transición hacia procesamiento distribuido
Las pruebas realizadas en este notebook demostraron que la generación de variables temporales sobre el dataset completo supera las capacidades de procesamiento de pandas en un entorno local.<br>
Aunque la lógica de transformación fue validada correctamente utilizando un subconjunto representativo, el volumen total del dataset requiere un enfoque distribuido.<br>
Por este motivo, las transformaciones diseñadas en este notebook fueron posteriormente implementadas utilizando Apache Spark, permitiendo paralelizar las operaciones y procesar el dataset completo de forma eficiente.